# MOUNT GOOGLE DRIVE

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# IMPORTS

In [18]:
import os, re, cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from glob import glob
from sklearn.metrics import confusion_matrix

# PATHS

In [19]:
BASE_PATH     = "/content/drive/MyDrive/Project work/Dataset/lgg-mri-segmentation/lgg_structured"
TRAIN_IMG_DIR = os.path.join(BASE_PATH, "train/images")
TRAIN_MSK_DIR = os.path.join(BASE_PATH, "train/masks")
VAL_IMG_DIR   = os.path.join(BASE_PATH, "val/images")
VAL_MSK_DIR   = os.path.join(BASE_PATH, "val/masks")
TEST_IMG_DIR  = os.path.join(BASE_PATH, "test/images")
TEST_MSK_DIR  = os.path.join(BASE_PATH, "test/masks")
MODEL_DIR     = "/content/drive/MyDrive/Project work/models/Segmentation"
os.makedirs(MODEL_DIR, exist_ok=True)

# CONFIG

In [20]:
IMG_SIZE   = 256
BATCH_SIZE = 16
EPOCHS     = 100
SEED       = 42
AUTOTUNE   = tf.data.AUTOTUNE

# LOAD FILE LISTS

In [21]:
def nsort(p):
    return [int(n) for n in re.findall(r"\d+", os.path.basename(p))]

train_imgs = sorted(glob(os.path.join(TRAIN_IMG_DIR, "*")), key=nsort)
train_msks = sorted(glob(os.path.join(TRAIN_MSK_DIR, "*")), key=nsort)
val_imgs   = sorted(glob(os.path.join(VAL_IMG_DIR,   "*")), key=nsort)
val_msks   = sorted(glob(os.path.join(VAL_MSK_DIR,   "*")), key=nsort)
test_imgs  = sorted(glob(os.path.join(TEST_IMG_DIR,  "*")), key=nsort)
test_msks  = sorted(glob(os.path.join(TEST_MSK_DIR,  "*")), key=nsort)

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")
assert len(train_imgs) == len(train_msks)
assert len(val_imgs)   == len(val_msks)

Train: 3133 | Val: 409 | Test: 387


# PREPROCESSING

In [22]:
def load_image(path):
    img = cv2.imread(path.numpy().decode(), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype(np.float32) / 255.0
    # CLAHE contrast enhancement — helps model see tumour boundaries clearly
    img_uint8 = (img * 255).astype(np.uint8)
    clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_uint8 = clahe.apply(img_uint8)
    img       = img_uint8.astype(np.float32) / 255.0
    return np.expand_dims(img, -1)

def load_mask(path):
    mask = cv2.imread(path.numpy().decode(), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return np.zeros((IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return np.expand_dims((mask > 127).astype(np.float32), -1)

def parse_sample(img_path, msk_path):
    img  = tf.py_function(load_image, [img_path], tf.float32)
    mask = tf.py_function(load_mask,  [msk_path], tf.float32)
    img.set_shape([IMG_SIZE, IMG_SIZE, 1])
    mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img, mask

# AUGMENTATION

In [23]:
# Aggressive but safe — all ops preserve mask alignment
def augment(img, mask):
    # Flips
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    # Rotation
    k    = tf.random.uniform([], 0, 4, dtype=tf.int32)
    img  = tf.image.rot90(img,  k)
    mask = tf.image.rot90(mask, k)
    # Brightness / contrast — image only
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    # Re-binarize mask after any float drift
    mask = tf.cast(mask > 0.5, tf.float32)
    return img, mask

def add_deep_supervision(img, mask):
    return img, {"final_out": mask, "d3_out": mask, "d2_out": mask}

# DATASETS

In [24]:
train_ds = (tf.data.Dataset.from_tensor_slices((train_imgs, train_msks))
            .shuffle(len(train_imgs), seed=SEED)
            .map(parse_sample, num_parallel_calls=AUTOTUNE)
            .map(augment,      num_parallel_calls=AUTOTUNE)
            .map(add_deep_supervision)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((val_imgs, val_msks))
          .map(parse_sample)
          .map(add_deep_supervision)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

# MODEL — ATTENTION U-NET

In [25]:
# Attention gates focus on tumour regions → big dice boost over plain U-Net

def conv_block(x, filters, dropout=0.0):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False,
                                kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    if dropout:
        x = tf.keras.layers.Dropout(dropout)(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False,
                                kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    return x

def attention_gate(x, g, filters):
    """
    x = skip connection (from encoder)
    g = gating signal   (from decoder)
    Outputs attended skip features — suppresses background noise.
    """
    Wx = tf.keras.layers.Conv2D(filters, 1, padding="same", use_bias=False)(x)
    Wg = tf.keras.layers.Conv2D(filters, 1, padding="same", use_bias=False)(g)
    # Upsample g to match x spatial size
    Wg = tf.keras.layers.UpSampling2D(size=(
        x.shape[1] // g.shape[1],
        x.shape[2] // g.shape[2]
    ))(Wg)
    psi = tf.keras.layers.Activation("relu")(
        tf.keras.layers.Add()([Wx, Wg])
    )
    psi = tf.keras.layers.Conv2D(1, 1, padding="same", use_bias=False)(psi)
    psi = tf.keras.layers.Activation("sigmoid")(psi)
    return tf.keras.layers.Multiply()([x, psi])

def build_attention_unet(img_size):
    inp = tf.keras.Input((img_size, img_size, 1))

    # Encoder
    c1 = conv_block(inp, 64);         p1 = tf.keras.layers.MaxPooling2D()(c1)
    c2 = conv_block(p1,  128);        p2 = tf.keras.layers.MaxPooling2D()(c2)
    c3 = conv_block(p2,  256);        p3 = tf.keras.layers.MaxPooling2D()(c3)
    c4 = conv_block(p3,  512);        p4 = tf.keras.layers.MaxPooling2D()(c4)

    # Bottleneck
    b  = conv_block(p4, 1024, dropout=0.4)

    # Decoder with attention gates on skip connections
    u1 = tf.keras.layers.Conv2DTranspose(512, 2, strides=2, padding="same")(b)
    a1 = attention_gate(c4, b,  512)
    u1 = tf.keras.layers.Concatenate()([u1, a1])
    u1 = conv_block(u1, 512)

    u2 = tf.keras.layers.Conv2DTranspose(256, 2, strides=2, padding="same")(u1)
    a2 = attention_gate(c3, u1, 256)
    u2 = tf.keras.layers.Concatenate()([u2, a2])
    u2 = conv_block(u2, 256)

    # Deep supervision at u2 scale
    d2_out = tf.keras.layers.Conv2D(1, 1, activation="sigmoid", name="d2_out")(
        tf.keras.layers.UpSampling2D(4)(u2))

    u3 = tf.keras.layers.Conv2DTranspose(128, 2, strides=2, padding="same")(u2)
    a3 = attention_gate(c2, u2, 128)
    u3 = tf.keras.layers.Concatenate()([u3, a3])
    u3 = conv_block(u3, 128)

    # Deep supervision at u3 scale
    d3_out = tf.keras.layers.Conv2D(1, 1, activation="sigmoid", name="d3_out")(
        tf.keras.layers.UpSampling2D(2)(u3))

    u4 = tf.keras.layers.Conv2DTranspose(64, 2, strides=2, padding="same")(u3)
    a4 = attention_gate(c1, u3, 64)
    u4 = tf.keras.layers.Concatenate()([u4, a4])
    u4 = conv_block(u4, 64)

    final_out = tf.keras.layers.Conv2D(1, 1, activation="sigmoid", name="final_out")(u4)

    return tf.keras.Model(inp, [final_out, d3_out, d2_out])

model = build_attention_unet(IMG_SIZE)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 256, 256,  │        576 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_30[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_26       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 256, 256,  │     36,864 │ activation_26[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        256 │ conv2d_31[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_27       │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 128, 128,  │          0 │ activation_27[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 128, 128,  │     73,728 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_32[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_28       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 128, 128,  │    147,456 │ activation_28[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        512 │ conv2d_33[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_29       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 64, 64,    │          0 │ activation_29[0]… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 64, 64,    │    294,912 │ max_pooling2d_5[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │      1,024 │ conv2d_34[0][0] 

 Total params: 32,094,083 (122.43 MB)

 Trainable params: 32,082,307 (122.38 MB)

 Non-trainable params: 11,776 (46.00 KB)

# LOSSES

In [26]:
# Tversky loss penalises false negatives more — critical for small tumours
def tversky(y_true, y_pred, alpha=0.3, beta=0.7, smooth=1.0):
    """alpha=FP weight, beta=FN weight. beta>alpha = recall-focused."""
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])
    TP = tf.reduce_sum(y_true * y_pred)
    FP = tf.reduce_sum((1 - y_true) * y_pred)
    FN = tf.reduce_sum(y_true * (1 - y_pred))
    return (TP + smooth) / (TP + alpha * FP + beta * FN + smooth)

def tversky_loss(y_true, y_pred):
    return 1.0 - tversky(y_true, y_pred)

def focal_tversky_loss(y_true, y_pred, gamma=0.75):
    """Focal variant — focuses training on hard examples."""
    tv = tversky(y_true, y_pred)
    return tf.pow(1.0 - tv, gamma)

def dice_coef(y_true, y_pred, smooth=1.0):
    """Metric only — not used as loss."""
    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def combined_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    ftl = focal_tversky_loss(y_true, y_pred)
    return bce + ftl

# COMPILE

In [27]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss={
        "final_out": combined_loss,
        "d3_out":    combined_loss,
        "d2_out":    combined_loss,
    },
    loss_weights={"final_out": 0.6, "d3_out": 0.3, "d2_out": 0.1},
    metrics={"final_out": dice_coef}
)

# CALLBACKS

In [28]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        ckpt_path,
        monitor="val_final_out_dice_coef",
        mode="max", save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_final_out_dice_coef",
        mode="max",                          # <-- this was missing
        patience=15, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_final_out_dice_coef",
        mode="max",                          # <-- add here too
        factor=0.5, patience=6, min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.TerminateOnNaN()
]

# PHASE 1 — TRAIN WITH Adam(3e-4)

In [29]:
print("\n--- Phase 1: Main Training ---")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

best_dice = max(history.history["val_final_out_dice_coef"])
print(f"\nPhase 1 Best Val Dice: {best_dice:.4f}")


--- Phase 1: Main Training ---
Epoch 1/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - d2_out_loss: 1.0989 - d3_out_loss: 1.1804 - final_out_dice_coef: 0.0436 - final_out_loss: 1.1746 - loss: 1.1688
Epoch 1: val_final_out_dice_coef improved from -inf to 0.02411, saving model to /content/drive/MyDrive/Project work/models/Segmentation/best_model.keras
196/196 ━━━━━━━━━━━━━━━━━━━━ 91s 332ms/step - d2_out_loss: 1.0984 - d3_out_loss: 1.1798 - final_out_dice_coef: 0.0438 - final_out_loss: 1.1740 - loss: 1.1682 - val_d2_out_loss: 1.1241 - val_d3_out_loss: 1.0678 - val_final_out_dice_coef: 0.0241 - val_final_out_loss: 1.1248 - val_loss: 1.1080 - learning_rate: 3.0000e-04
Epoch 2/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - d2_out_loss: 0.8870 - d3_out_loss: 0.9078 - final_out_dice_coef: 0.1346 - final_out_loss: 0.9401 - loss: 0.9251
Epoch 2: val_final_out_dice_coef improved from 0.02411 to 0.11589, saving model to /content/drive/MyDrive/Project work/models/Segmentation/best_model.keras
1

KeyboardInterrupt: 

# PHASE 2 — FINE-TUNE AT VERY LOW LR

In [ ]:
# Even if best_dice > 0.80, fine-tune squeezes out extra %
print("\n--- Phase 2: Fine-Tune at lr=1e-5 ---")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss={
        "final_out": combined_loss,
        "d3_out":    combined_loss,
        "d2_out":    combined_loss,
    },
    loss_weights={"final_out": 0.6, "d3_out": 0.3, "d2_out": 0.1},
    metrics={"final_out": dice_coef}
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

# PLOT

In [ ]:
all_train = history.history["final_out_dice_coef"] + history2.history["final_out_dice_coef"]
all_val   = history.history["val_final_out_dice_coef"] + history2.history["val_final_out_dice_coef"]

plt.figure(figsize=(10, 4))
plt.plot(all_train, label="Train Dice")
plt.plot(all_val,   label="Val Dice")
plt.axhline(0.80, color="red", linestyle="--", label="Target 0.80")
plt.axvline(len(history.history["final_out_dice_coef"]), color="gray",
            linestyle=":", label="Fine-tune start")
plt.title("Dice Score — All Phases")
plt.xlabel("Epoch"); plt.ylabel("Dice")
plt.legend(); plt.tight_layout(); plt.show()

# TEST EVALUATION WITH TTA

In [ ]:
# Test-Time Augmentation: average predictions over flips → free +1-2% dice
def predict_with_tta(model, imgs):
    p_orig  = model.predict(imgs,                          verbose=0)[0]
    p_hflip = model.predict(tf.image.flip_left_right(imgs),verbose=0)[0]
    p_vflip = model.predict(tf.image.flip_up_down(imgs),   verbose=0)[0]
    p_hflip = tf.image.flip_left_right(p_hflip).numpy()
    p_vflip = tf.image.flip_up_down(p_vflip).numpy()
    return (p_orig + p_hflip + p_vflip) / 3.0

print("\nTest set evaluation (with TTA):")
y_true_all, y_pred_all = [], []

test_ds_raw = (tf.data.Dataset.from_tensor_slices((test_imgs, test_msks))
               .map(parse_sample).batch(BATCH_SIZE))

for imgs, masks in test_ds_raw:
    preds = predict_with_tta(model, imgs)
    y_true_all.extend(masks.numpy().flatten().astype(np.uint8))
    y_pred_all.extend((preds > 0.5).flatten().astype(np.uint8))

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
cm = confusion_matrix(y_true_all, y_pred_all)
TN, FP, FN, TP = cm.ravel()

final_dice = (2 * TP) / (2 * TP + FP + FN + 1e-8)
print("\n=== FINAL SEGMENTATION METRICS ===")
print(f"Dice Score     : {final_dice:.4f}  {'✓ TARGET MET' if final_dice >= 0.80 else '✗ below target'}")
print(f"IoU Score      : {TP / (TP + FP + FN + 1e-8):.4f}")
print(f"Precision      : {TP / (TP + FP + 1e-8):.4f}")
print(f"Recall         : {TP / (TP + FN + 1e-8):.4f}")
print(f"Pixel Accuracy : {(TP + TN) / (TP + TN + FP + FN + 1e-8):.4f}")

plt.figure(figsize=(5, 5))
plt.imshow(cm, cmap="Blues"); plt.title("Pixel Confusion Matrix"); plt.colorbar()
plt.xticks([0, 1], ["Background", "Tumor"])
plt.yticks([0, 1], ["Background", "Tumor"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

# SAVE

In [ ]:
model.save(os.path.join(MODEL_DIR, "brain_segmentation.keras"))
print("\nModel saved to:", os.path.join(MODEL_DIR, "brain_segmentation.keras"))